In [2]:
from pathlib import Path
import numpy as np
from typing import Literal, Optional
import os
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
import matplotlib.pyplot as plt
import seaborn as sns


project_root = Path(".").resolve().parent.parent.parent
assert project_root.name == "pep-compass"

In [3]:
import pandas as pd

parents_negative = pd.read_csv(
    project_root
    / "results/data/hydramp/dbaasp/all/dbaasp_clean_leq25.csv"
)
mutants_negative = pd.read_csv(
    project_root
    / "results/mutants/mutants/hydramp/dbaasp/all/dbaasp_clean_parentleq25_direction_threshold=1e-06_token_threshold=0.1_jacobian_mode=approx_jacobian_eps=0.05_leq25.csv"
)

parents_positive = pd.read_csv(
    project_root
    / "results/data/hydramp/dbaasp/all/high_ecoli_activity_leq25.csv"
)
mutants_positive = pd.read_csv(
    project_root
    / "results/mutants/mutants/hydramp/dbaasp/all/high_ecoli_activity_parentleq25_direction_threshold=1e-06_token_threshold=0.1_jacobian_mode=approx_jacobian_eps=0.05_leq25.csv"
    )

In [4]:
def compute_diff(
    parents_df: pd.DataFrame,
    mutants_df: pd.DataFrame,
    match_col_parents: str,
    match_col_mutants: str,
    value_cols: list[str],
    value_preprocessing: Optional[Literal["log"]],
    add_relative: bool,
) -> pd.DataFrame:
    """
    Compute the difference between the mutants and parents for each value column.
    """
    merged = mutants_df.merge(
        parents_df,
        left_on=match_col_mutants,
        right_on=match_col_parents,
        suffixes=("_mutants", "_parents"),
    )
    if value_preprocessing is None:
        vprep_fn = lambda x: x
    elif value_preprocessing == "log":
        vprep_fn = np.log2
    else:
        raise ValueError(f"Invalid value_preprocessing: {value_preprocessing}")
    if value_preprocessing is None:
        value_preprocessing = ""
    else:
        value_preprocessing = "_" + value_preprocessing

    for value_col in value_cols:
        merged[f"{value_col}{value_preprocessing}_diff"] = vprep_fn(
            merged[f"{value_col}_mutants"]
        ) - vprep_fn(merged[f"{value_col}_parents"])
        if add_relative:
            merged[f"{value_col}{value_preprocessing}_diff_relative"] = merged[
                f"{value_col}{value_preprocessing}_diff"
            ] / vprep_fn(merged[f"{value_col}_parents"])
    return merged


In [5]:
mic_bac_columns = parents_negative.head().columns.tolist()[3:]

In [6]:
diff_negative = compute_diff(
    parents_df=parents_negative,
    mutants_df=mutants_negative,
    match_col_parents="description",
    match_col_mutants="parent_id",
    value_cols=mic_bac_columns,
    value_preprocessing="log",
    add_relative=True,
)
diff_positive = compute_diff(
    parents_df=parents_positive,
    mutants_df=mutants_positive,
    match_col_parents="description",
    match_col_mutants="parent_id",
    value_cols=mic_bac_columns,
    value_preprocessing="log",
    add_relative=True,
)

ALL_AA = list("ACDEFGHIKLMNPQRSTVWY")

diff_clean_positive = diff_positive.dropna(subset=["parent", "mutant", "position"]).drop_duplicates(subset=["parent", "mutant", "position"])


mask = (
    (diff_clean_positive["position"] < diff_clean_positive["parent"].str.len()) &
    (diff_clean_positive["position"] < diff_clean_positive["mutant"].str.len())
)
diff_clean_positive = diff_clean_positive[mask]


diff_clean_negative = diff_negative.dropna(subset=["parent", "mutant", "position"]).drop_duplicates(subset=["parent", "mutant", "position"])


mask = (
    (diff_clean_negative["position"] < diff_clean_negative["parent"].str.len()) &
    (diff_clean_negative["position"] < diff_clean_negative["mutant"].str.len())
)
diff_clean_negative = diff_clean_negative[mask]

In [7]:
#merge diffs into one but leave label 0 negative 1 postive
diff_clean_negative['label'] = 0
diff_clean_positive['label'] = 1
diff_clean = pd.concat([diff_clean_negative, diff_clean_positive], ignore_index=True)



In [8]:
diff_clean

,mutant,position,parent,parent_id,A. baumannii ATCC 19606_mutants,E. coli ATCC 11775_mutants,E. coli AIG221_mutants,E. coli AIG222_mutants,K. pneumoniae ATCC 13883_mutants,P. aeruginosa PA01_mutants,...,E. coli Nissle_log_diff_relative,Salmonella enterica ATCC 9150 (BEIRES NR-515)_log_diff,Salmonella enterica ATCC 9150 (BEIRES NR-515)_log_diff_relative,Salmonella enterica (BEIRES NR-170)_log_diff,Salmonella enterica (BEIRES NR-170)_log_diff_relative,Salmonella enterica ATCC 9150 (BEIRES NR-174)_log_diff,Salmonella enterica ATCC 9150 (BEIRES NR-174)_log_diff_relative,L. monocytogenes ATCC 19111 (BEIRES NR-106)_log_diff,L. monocytogenes ATCC 19111 (BEIRES NR-106)_log_diff_relative,label
0,FLGLLFHGVHHVGKWIHGNIHGHH,18,FLGLLFHGVHHVGKWIHGLIHGHH,DBAASP_1001,83.673540,113.194336,105.330060,76.993530,59.806360,55.607685,...,-0.008496,0.383081,0.082083,0.178596,0.018772,0.184985,0.023641,0.330948,0.064074,0
1,FLGLLFHGVHHVGKWIHGIIHGHH,18,FLGLLFHGVHHVGKWIHGLIHGHH,DBAASP_1001,76.766450,109.609700,101.310745,74.697845,50.966095,46.862800,...,-0.018160,0.040210,0.008616,-0.007879,-0.000828,-0.066927,-0.008553,0.051298,0.009932,0
2,FLGLLFHGVHHVGKWIHGEIHGHH,18,FLGLLFHGVHHVGKWIHGLIHGHH,DBAASP_1001,104.181110,125.732240,118.488990,98.296370,80.257400,75.969790,...,0.069494,0.834084,0.178719,0.424473,0.044616,0.654384,0.083631,0.735229,0.142345,0
3,FLGLLFHGVHHVGKWIHGKIHGHH,18,FLGLLFHGVHHVGKWIHGLIHGHH,DBAASP_1001,66.921820,93.771110,87.084960,72.269580,37.082706,35.130596,...,-0.066390,-0.432021,-0.092569,-0.022259,-0.002340,-0.150654,-0.019254,-0.370305,-0.071693,0
4,FLGLLFHGVHHVGKWIHGLIVGHH,20,FLGLLFHGVHHVGKWIHGLIHGHH,DBAASP_1001,77.838580,113.621750,104.875145,75.585450,56.882526,51.541615,...,0.008679,0.231504,0.049604,0.079699,0.008377,0.143732,0.018369,0.206367,0.039954,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106636,LLPWKWIWWKWRR,6,LLPWKWPWWKWRR,DBAASP_10694,42.117832,99.160650,91.479034,61.479840,40.959923,17.017202,...,0.000289,-0.699782,-0.132337,-0.175068,-0.019900,-0.300381,-0.034629,-0.618300,-0.106492,1
106637,LLPWKWKWWKWRR,6,LLPWKWPWWKWRR,DBAASP_10694,56.130257,74.791730,76.946630,90.852234,49.359818,25.131464,...,0.006494,-0.403659,-0.076336,0.027925,0.003174,0.059736,0.006887,-0.313172,-0.053939,1
106638,LLPWKWQWWKWRR,6,LLPWKWPWWKWRR,DBAASP_10694,77.834730,99.605640,95.159950,99.039810,64.883950,38.661720,...,0.076445,0.161648,0.030569,0.361439,0.041085,0.023906,0.002756,-0.014833,-0.002555,1
106639,LLPWKWFWWKWRR,6,LLPWKWPWWKWRR,DBAASP_10694,50.260666,106.336750,93.892250,76.810200,46.730070,20.388280,...,0.012148,-0.505391,-0.095575,-0.127701,-0.014516,-0.206585,-0.023816,-0.469383,-0.080843,1


In [9]:
diff_clean['transition'] = diff_clean.apply(
    lambda row: row['mutant'][int(row['position'])] + row['parent'][int(row['position'])],
    axis=1
)


In [10]:
y = diff_clean['E. coli ATCC 11775_log_diff']

In [11]:
transition_dummies = pd.get_dummies(diff_clean['transition'], prefix='trans', drop_first=True)


In [12]:
transition_dummies

,trans_AD,trans_AE,trans_AF,trans_AG,trans_AH,trans_AI,trans_AK,trans_AL,trans_AM,trans_AN,...,trans_YK,trans_YL,trans_YM,trans_YN,trans_YP,trans_YQ,trans_YR,trans_YT,trans_YV,trans_YW
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106636,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
106637,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
106638,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
106639,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [13]:
# transition_dummies = transition_dummies.mul(diff_clean['label'], axis=0)


In [14]:
from sklearn.linear_model import LinearRegression
import numpy as np
import pandas as pd
import statsmodels.api as sm

# X and y
X = transition_dummies.astype(float).values  # no transpose
y = diff_clean['E. coli ATCC 11775_log_diff'].values

# Fit OLS model
X_sm = sm.add_constant(X)  # Adds intercept
ols_model = sm.OLS(y, X_sm)
results = ols_model.fit()

# Extract coefficients & p-values (excluding constant)
coef = results.params[1:]      # pomiń constant
pvals = results.pvalues[1:]    # pomiń constant

model_coef_df = pd.DataFrame({
    "aa_change": transition_dummies.columns,
    "coefficient": coef,
    "p_value": pvals
})

# Display
print(model_coef_df.head())


  aa_change  coefficient   p_value
0  trans_AD    -0.138556  0.067014
1  trans_AE    -0.079255  0.339676
2  trans_AF     0.113565  0.145882
3  trans_AG     0.062853  0.401358
4  trans_AH     0.077217  0.426450


In [15]:
from statsmodels.stats.multitest import multipletests

In [16]:
# --- Apply Benjamini–Hochberg correction (FDR)
reject, pvals_corrected, _, _ = multipletests(model_coef_df["p_value"], method="fdr_bh")

# Add to DataFrame
model_coef_df["p_adj"] = pvals_corrected
model_coef_df["significant_FDR_5%"] = reject

# --- Sort by adjusted p-value for clarity
model_coef_df = model_coef_df.sort_values("p_adj")

In [17]:
model_coef_df[model_coef_df["significant_FDR_5%"]!=True]

,aa_change,coefficient,p_value,p_adj,significant_FDR_5%
39,trans_DE,0.183856,0.014992,0.052471,False
290,trans_SM,0.272294,0.015819,0.054341,False
351,trans_WT,-0.254928,0.015729,0.054341,False
117,trans_HF,0.180833,0.016224,0.055222,False
71,trans_ET,0.182866,0.016803,0.055664,False
...,...,...,...,...,...
160,trans_KM,-0.006897,0.963146,0.973644,False
368,trans_YT,0.006788,0.969066,0.976966,False
85,trans_FN,0.000751,0.992451,0.996687,False
203,trans_MT,0.000939,0.994001,0.996687,False


In [18]:
# Wyświetl tylko kolumny, które Cię interesują



print(model_coef_df[["aa_change", "p_value", "p_adj", "significant_FDR_5%"]].head(100))


    aa_change       p_value         p_adj  significant_FDR_5%
50   trans_DR  3.508239e-25  1.301557e-22                True
25   trans_CK  3.158937e-18  5.859828e-16                True
44   trans_DK  1.385494e-17  1.713394e-15                True
64   trans_EL  4.134395e-16  3.834652e-14                True
43   trans_DI  2.030947e-14  1.255802e-12                True
..        ...           ...           ...                 ...
211  trans_NF  9.260378e-03  3.578750e-02                True
360  trans_YI  9.557333e-03  3.655434e-02                True
282  trans_SD  9.773934e-03  3.689639e-02                True
235  trans_PN  9.845667e-03  3.689639e-02                True
237  trans_PR  1.029165e-02  3.818204e-02                True

[100 rows x 4 columns]


In [19]:
significant = model_coef_df.loc[model_coef_df["p_adj"] < 0.1, 
                                ["aa_change", "coefficient", "p_value", "p_adj", "significant_FDR_5%"]]
print(significant.sort_values("p_adj"))


    aa_change  coefficient       p_value         p_adj  significant_FDR_5%
50   trans_DR     0.815313  3.508239e-25  1.301557e-22                True
25   trans_CK     0.696167  3.158937e-18  5.859828e-16                True
44   trans_DK     0.659426  1.385494e-17  1.713394e-15                True
64   trans_EL     0.609080  4.134395e-16  3.834652e-14                True
43   trans_DI     0.682415  2.030947e-14  1.255802e-12                True
..        ...          ...           ...           ...                 ...
299  trans_TA     0.168459  3.307733e-02  9.367702e-02               False
248  trans_QG     0.189081  3.283188e-02  9.367702e-02               False
276  trans_RT    -0.318186  3.304468e-02  9.367702e-02               False
103  trans_GM     0.162839  3.362398e-02  9.450376e-02               False
30   trans_CQ     0.264665  3.407860e-02  9.506135e-02               False

[133 rows x 5 columns]


In [22]:
results.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.400
Model:                            OLS   Adj. R-squared:                  0.398
Method:                 Least Squares   F-statistic:                     191.2
Date:                Sun, 09 Nov 2025   Prob (F-statistic):               0.00
Time:                        20:42:35   Log-Likelihood:                 8458.8
No. Observations:              106641   AIC:                        -1.617e+04
Df Residuals:                  106269   BIC:                        -1.261e+04
Df Model:                         371                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0743      0.075     -0.996      0.319      -0.221       0.072
x1            -0.1386      0.076     -1.832      0.067      -0.287       0.010
x2            -0.0793      0.083     -0.955      0.340      -0.242       0.083
x3             0.1136      0.078      1.454      0.146      -0.039       0.267
x4             0.0629      0.075      0.839      0.401      -0.084       0.210
x5             0.0772      0.097      0.795      0.426      -0.113       0.268
x6             0.2287      0.080      2.858      0.004       0.072       0.386
x7             0.3696      0.076      4.839      0.000       0.220       0.519
x8             0.1650      0.080      2.061      0.039       0.008       0.322
x9             0.1334      0.076      1.752      0.080      -0.016       0.283
x10            0.0932      0.091      1.028      0.304      -0.084       0.271
x11            0.0613      0.106      0.581      0.561      -0.146       0.268
x12            0.0644      0.092      0.698      0.485      -0.117       0.245
x13            0.4655      0.078      5.986      0.000       0.313       0.618
x14            0.0796      0.075      1.062      0.288      -0.067       0.227
x15           -0.0047      0.091     -0.052      0.958      -0.182       0.173
x16            0.1070      0.079      1.347      0.178      -0.049       0.263
x17            0.1804      0.083      2.162      0.031       0.017       0.344
x18            0.0140      0.109      0.128      0.898      -0.199       0.227
x19            0.4112      0.090      4.575      0.000       0.235       0.587
x20            0.0453      0.149      0.303      0.762      -0.247       0.338
x21            0.0864      0.109      0.794      0.427      -0.127       0.300
x22            0.3011      0.075      4.019      0.000       0.154       0.448
x23            0.2282      0.078      2.929      0.003       0.076       0.381
x24            0.2190      0.075      2.920      0.004       0.072       0.366
x25            0.2676      0.076      3.505      0.000       0.118       0.417
x26            0.6962      0.080      8.707      0.000       0.539       0.853
x27            0.3485      0.078      4.476      0.000       0.196       0.501
x28            0.2097      0.236      0.888      0.374      -0.253       0.672
x29            0.1790      0.093      1.919      0.055      -0.004       0.362
x30            0.2196      0.080      2.743      0.006       0.063       0.376
x31            0.2647      0.125      2.119      0.034       0.020       0.509
x32            0.5961      0.078      7.659      0.000       0.444       0.749
x33            0.1375      0.082      1.676      0.094      -0.023       0.298
x34            0.1081      0.078      1.390      0.165      -0.044       0.260
x35            0.2098      0.075      2.794      0.005       0.063       0.357
x3